<a href="https://colab.research.google.com/github/tuankhoin/CO3057-Computer-Vision/blob/main/Week_12_Object_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ho Chi Minh University of Technology (HCMUT)

CO3057 - Digital Image Processing and Computer Vision

# Week 12 - Object Detection





<iframe width="560" height="315" src="https://www.youtube.com/embed/WZmSMkK9VuA?si=9MXvwObKnFJguN9m" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" allowfullscreen></iframe>

## Learning Outcomes
- Explain the standard approach to object detection (sliding window)
- Explain how object detection is implemented in region-based CNNs
- Explain object detection methods
- Explain design issues and trade-offs involved in building detection methods


---
# Object Detection Basics
---

## Classification vs. Detection

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/classification-v-detection.png?raw=true)

- Object detection = locate objects in an image
	- Classification: "Is this a $<\text{class label}>$?"
	- Detection: "Where is the $<\text{class label}>$?"
- Object detection is usually modelled as a classification task performed within patches of an image
	- Detection: For every patch, "Is this a $<\text{class label}>$?"

## Window Evaluation: IoU
- Common method to evaluate a window result is Intersection over Union (IoU) between true bounding box and detection window

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/iou-formula.png?raw=true)

## Example: IoU
- Our predicted box is $3 \times 7= 21$
- The overlap that we have with the target box is $18$
- The total area of union is the area of total area of the box, $6 \times 8 + 3 = 51$
- $IoU = 18/51 = 0.35$

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/example-iou.png?raw=true)

## Summary
- Object detection is generally modelled as image classification within small regions of an image
- Windows over some threshold = "detections", can be evaluated using IoU with ground truth
- Problems:
	- Very large number of possible windows (slow, increases probability of false detections)
	- Overall evaluation of images with multiple targets can be complicated (multiple targets, multiple detection windows, different IoUs)



---
# R-CNN
---

## R-CNN
- R-CNN = Region-based convolutional neural network
- Given an image, identify a small number of windows for object detection ("region proposals" or "region of interest (ROIs)")

## Generating Region Proposals
- R-CNN uses Selective Search to generate ROIs
- Selective Search algorithm:
	- Oversegment image into superpixels
	- Iteratively combine adjacent superpixels based on similarity in colour + texture, size, and compactness

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/generating_region_proposals.png?raw=true)

## R-CNN: Region-Based CNN
- We first take an image

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/rcnn-region-based-cnn.png?raw=true)

- We take a bunch of regions of interest that or selective search selected (approximately 2000 in total)

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/region-based-cnn.png?raw=true)

- Since our regions of interest may not necessarily be square but various dimensions, we will then warp these patches to become square

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/r-cnn-region-based-cnn.png?raw=true)

- We then run each of these patches through a **single** CNN

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/region-based-cnn-2.png?raw=true)

- The CNN then predicts both the class of the patch AND the bbox of that class
- We will also perform regression with the predicted bbox on the ground truth bbox to teach the CNN to learn how to form the box around the object
- The factors of the bbox that it needs to learn are the transformed image's centre $x,y$ coordinates, as well as the box's height and width

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/region-based-cnn-3.png?raw=true)

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/cnn-region-based-cnn.png?raw=true)

## Bounding Box Computation
- Original region proposal = ($p_x, p_y, p_h, p_w$)
- Transform = ($t_x, t_y, t_h, t_w$)
	- Remember: this is just a factor of the original region proposal box so this needs to be rescaled to the actual dimensions of the image
- Goal: compute bounding box = ($b_x, b_y, b_h, b_w$)
- Step 1. Translate

$$
b_x = p_x + p_w t_x, \ \  \ \ b_y = p_y + p_h t_y
$$

- Step 2. Scale

$$
b_w = p_w \exp(t_w), \ \ \ \ \ b_h = p_h \exp(t_h)
$$

## R-CNN Training
- CNN pretrained on ImageNet
- Last layer (1x1000) is replaced with a new classification layer of size 1x(N+1)
	- $N+1$ = $N$  object classes + "background" class, i.e. the $(N+1)\text{th}$ class is _not interesting_ to us
	- CNN is retrained on (N+1)-way detection, using regions with IoU >= 0.5 as ground truth "objects"
	- Sample regions so 75% of training set is "background"
- CNN features are used as input to:
	- Label classification model (1-vs-all linear SVM)
	- Bounding box model (class-specific linear regression)

## R-CNN Testing
- Input test image
- Compute region proposals (Selective Search)
- Each region: run through CNN to predict class labels and bounding box transforms
- "Detections" = regions with highest confidence scores
	- Based on a threshold, or top $k$?
	- Overall, or pre-category?
	- These are design choices that we can make
		- Often we just use threshold

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/r-cnn-testing.png?raw=true)

## R-CNN Results (Random Sample)

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/r-cnn-results.png?raw=true)

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/rcnn-results-authors-pick.png?raw=true)

## R-CNN Summary
- R-CNN (Region-based convolutional neural network) does classification in parallel over a set of region proposals
- Output: class labels and bounding box transforms
- Advantages
	- Much more efficient than classifying every window
- Disadvantages
	- Still requires classifying many windows (e.g., 2000)
	- Region proposal step could miss some objects




---
# Fast R-CNN
---

## Fast R-CNN
- Major change: Run the whole image through a _reasonably shallow_ fully-convolutional neural network
- Take region proposals from last convolutional layer
- What this means is that rather than taking samples from the image like we did for R-CNN, we are instead going to take samples from the feature map output of the CNN
- This saves us on computations, as there are many, many overlapping proposals and therefore redundant calculations

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/fast-rcnn.png?raw=true)


## Mechanism

- Backbone will be used to extract features from the image, i.e. AlexNet or ResNet
- The per-region network doesn't need to be complex since the output is so downsized and processed

- What architecture to use for the "backbone" FCN and per-region networks?
	- AlexNet, ResNet, etc.

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/fast-rcnn8.png?raw=true)

## Fast R-CNN Training
- Train on $R$ regions sampled from $N$ images
	- For efficiency, $N$ is small and $R$ is large (e.g., 64)
	- Sample regions so 75% of training set is "background"
- Train with a multi-task loss: $L=L_{cls} + L_{loc}$
	- $L_{cls} =$ cross-entropy loss over labels
	- $L_{loc} =$ SmoothL1 of $\text{abs}(\text{true}-\text{predicted bbox parameter})$
		 - This is so we have an acceptable first derivative of the L1 loss, as there will be discontinuities when L1 = 0
 - $L_{loc}$ computed for object classes only
	- i.e. if it is background, then we won't compute the loss, we will only compute the loss over the actual object classes
![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/fast-rcnn-training.png?raw=true)

## Fast R-CNN Summary
- End-to-end region-based convolutional neural network
- Advantages:
	- Faster than R-CNN (~9x faster training, ~140x faster test)
	- Slightly more accurate than R-CNN
- Disadvantage:
	- ROIs aren't learned; region proposal step could miss some objects



---
# Faster R-CNN
---

## Faster R-CNN
- Major change: network learns region proposals, instead of Selective Search
- The left-hand side are the new parts for faster R-CNN - a region proposal network

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/faster-rcnn.png?raw=true)

## Region Proposal Network (RPN)
- Each pixel in our downsampled representation of the image gets the chance to propose regions of interest, i.e. anchor points.
- We let the anchor point make $k$ proposals of what region that it belongs to, as well as a fixed size and aspect ratio for the region
- In each region, predict object class and bounding box transform

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/region-proposal-network.png?raw=true)

## Faster R-CNN
- For each region:
	- Crop & resize features
	- Predict object class and bbox transform
- For each image:
	- Run backbone CNN to get feature map
	- Compute region proposals from RPN

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/faster-rcnn.png?raw=true)

## Faster R-CNN Training
- RPN loss is weighted sum of:
	- Classification loss: binary cross entropy loss (any object vs. background)
	- Regression loss: Smooth L1 between true and predicted bbox parameters
- As in Fast R-CNN, training samples $R$ regions (anchors) from; $N$ images ($R=256, N=1$)
	- Anchors are sampled so up to 50% are objects
- Full network is RPN + Fast R-CNN (sharing a backbone)
	- Various ways to train this, but original method alternatives between training RPN and Fast R-CNN
 - Generally first train Region Proposal Network, and then train the final layer that classifies bbox and class

## Faster R-CNN Summary
- Faster R-CNN is similar to Fast R-CNN but learns the region proposals with a region network (RPN)
- Even faster than fast R-CNN (~10x faster test)
- Modular approach - variations on Faster R-CNN with deep backbone tend to be quite accurate
	- Speed-accuracy trade-off: deeper networks are also slower

## Summary
- Object detection = classification of image regions
- Exhaustive search is slow; most methods use only a subset of regions ("region proposals" or "regions of interest (ROIs)")
- Many parameters to consider:
	- What counts as a true detection / true rejection (IoU threshold)?
	- How to select region proposals?
	- How to deal with class imbalance? ("background" is most common class)

---
# Evaluating Object Detectors
---

## Object Detection Result
- Typically, object detectors will return overlapping detections
	- Can be different objects, or the same object detected at multiple scales/positions
- Treat as multiple detections? Or select one as the final prediction?

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/object_detection_result.png?raw=true)

## Non-Max Suppression (NMS)
- Typical approach: non-maximum suppression (NMS)
- Algorithm:
	- Starting with the highest-scoring bounding-box...
	- Drop bounding boxes with lower score that overlap with this box above some IoU threshold (e.g., 0.7)
	- Repeat with next highest-scoring bounding box
- Often done separately within each object class

- Below we look for the bbox with the highest probability, which in this case is $P(dog)=0.9$.

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/non_max_suppression.png?raw=true)

- Since the other bbox heavily overlaps with this bbox (it has an IoU of 0.78 > 0.7), we remove it
- Now we look for the next bbox with the highest probability with overlapping bboxes, which is the RHS bbox
- Since we have an overlapping bbox with an IoU of 0.74, we remove the lower probability bbox

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/non-max-suppression2.png?raw=true)

- Now we are left with the two bboxes, since they have a low IoU we leave them as is

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/non-max-suppression3.png?raw=true)

- NMS can drop some correct detections when objects are highly overlapping
- In the below image, it's likely that bboxes would have an IoU greater than 0.7 and so we would lose information
- But generally this is preferable to counting the same object many times

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/nms-4.png?raw=true)

## Evaluation
- How to evaluate, given that there may be multiple objects/detections per image?
- Commonly-used method:
	- Run detection on entire test set
	- Run NMS to remove overlapping detections
	- For each object category, compute Average Precision (AP) = area under precision-recall (P-R) curve

- Below we have 5 dog detections and 3 ground truths which we will use as an example how how we calculate the AP score

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/evaluation-nms.png?raw=true)

- We first pick out our highest probability score and then check if it has an IoU > 0.5. If it does, we consider it a positive, else a negative

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/evaluation-nsm2.png?raw=true)

- Remember that $Precision = \frac{TP}{TP+FP}$ and $Recall = \frac{TP}{TP+FN}$,
- Another way we can think of this is $Precision = \frac{TP}{\text{Total Positives}}$ and $Recall = \frac{TP}{\text{Total Ground Truths}}$
- Therefore, we get $Precision = 1 / 1$ and $Recall = 1 / 3$
-  We plot this precision and recall on the below plot

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/evaluation-nms3.png?raw=true)

- We then pick the second highest score, which also has an IoU > 0.5
- We calculate the precision and recall as shown below and then plot it

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/evaluation-nms5.png?raw=true)

- For the next score, we get an IoU > 0.5, therefore we consider this a False Positive, increasing the denominator of Precision by 1, and thus making a lower score
- We plot these value on the below graph

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/evaluation-nms6.png?raw=true)

- Same for the next point, follow the same procedure

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/evaluation-nms7.png?raw=true)

- For the last one we got a positive and get a recall of 1.0 and precision of 0.6
- Plot the final point on the graph

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/evalutation-nms8.png?raw=true)

- To get our AP score, we need to find the area underneath this curve that we created
- In this case, we got a very good score

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/evaluation-nsm9.png?raw=true)

## Properties of P-R Curve
- What is the best possible AP (area under P-R curve)?
	- The best AP is 1.0!
- How would you accomplish this?
	- Get all precision scores as 1.0

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/properties-of-pr-curve.png?raw=true)

## Mean Average Precision (mAP)
- Example:
	- Bird AP = 0.65
	- Cat AP = 0.80
	- Dog AP = 0.86
	- mAP@0.5 = 0.77
		 - This is the average of the AP scores of each class
- "COCO mAP": Compute mAP for **multiple IoU thresholds** (0.5, 0.55, 0.6, ..., 0.95)
	- Example: mAP@0.5 = 0.77, mAP@0.55 = 0.72, ... mAP@0.95 = 0.19
	- COCO mAP = 0.45

## Evaluation Summary
- Common metric for evaluating object detectors is mAP (or COCO mAP)
- Both NMS and P-R steps require IoU thresholds; different thresholds can change results
- Object detection is complex - one number is not very informative
	- How accurate is the object classification?
	- How accurate are the bounding boxes?
	- What kind of errors is the model making (misses, false alarms)?
